# GPT

In [1]:
from google.colab import userdata
import os, json, time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

TODO: Add BDI-II items like this: 

BDI_II_ITEMS = {
    1: {
        "name": "",
        "options": {
            0: "",
            1: "",
            2: "",
            3: "",
        },
    },
    2: {
        ...
    }
}

Removed in this notebook due to copyright.

In [ ]:
BDI_II_ITEMS = {
    1: {
        "name": "x",
        "options": {
            0: "y",
            1: "z",
            2: "t",
            3: "v",
        },
    },
    
}

SYMPTOMS = {i: v["name"] for i, v in BDI_II_ITEMS.items()}

In [4]:
# Build item text for a subset of relevant items
def build_items_text(relevant_items: list[int]) -> str:
    """Return BDI-II scale text for only the relevant items."""
    lines = []
    for i in sorted(relevant_items):
        item = BDI_II_ITEMS[i]
        lines.append(f"{i}. {item['name']}:")
        for score, text in item["options"].items():
            lines.append(f"  {score}) {text}")
        lines.append("")
    return "\n".join(lines).strip()


# Prompt
def format_score_prompt(
    clinician_msg: str,
    patient_reply: str,
    relevant_items: list[int],
) -> str:
    items_text = build_items_text(relevant_items)
    return f"""You are a clinical language analysis assistant trained on the Beck Depression Inventory-II (BDI-II).

            The following BDI-II items have been identified as relevant to this exchange. Each item lists its possible score options (0–3):

            {items_text}

            ---

            The following is a conversational exchange between a clinician and a patient:

            Clinician: "{clinician_msg}"
            Patient: "{patient_reply}"

            ---

            Your task:
            For each BDI-II item listed above, select the score (0, 1, 2, or 3) that best matches the severity expressed in the patient's reply. Base your scoring strictly on the patient's words.

            Respond in JSON only — no extra text:
            {{
              "scores": {{
                "<item_number>": <score>,
                ...
              }}
            }}
            """


# API Call
def call_gpt_scores(
    clinician_msg: str,
    patient_reply: str,
    relevant_items: list[int],
    model: str = "gpt-4o-mini",
    max_retries: int = 3,
) -> str:
    if not relevant_items:
        return json.dumps({"scores": {}})
    wait = 5
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[{
                    "role": "user",
                    "content": format_score_prompt(clinician_msg, patient_reply, relevant_items),
                }],
                max_completion_tokens=300,
                response_format={"type": "json_object"},
            )
            return completion.choices[0].message.content.strip()
        except Exception as e:
            print(f"  Warning (attempt {attempt+1}/{max_retries}): {str(e)[:100]}")
            if attempt < max_retries - 1:
                time.sleep(wait)
                wait *= 2
            else:
                return f"[ERROR: {str(e)[:100]}]"


df = pd.read_csv("bdichat_turn_annotations.csv")
df = df.dropna(subset=["PersonaReply"]).reset_index(drop=True)
df["Message"] = df["Message"].fillna("")
# df = df.head(100)  # uncomment for a quick test run
print(f"Loaded {len(df)} rows.")

# Build relevant-items list per row from ground-truth labels
# Uses ground-truth symptom label columns ("1. Sadness", etc.).
# To use predicted labels instead, swap this with pred_label_i columns.
GT_SYMPTOM_COLS = [f"{i}. {SYMPTOMS[i]}" for i in range(1, 22)]

def get_relevant_items(row) -> list[int]:
    return [i for i in range(1, 22) if row[GT_SYMPTOM_COLS[i - 1]] == 1]


# Run GPT
raw_score_outputs = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="GPT score detection"):
    relevant = get_relevant_items(row)
    raw_score_outputs.append(
        call_gpt_scores(row["Message"], row["PersonaReply"], relevant)
    )

df["raw_score_output"] = raw_score_outputs

# Parse Output → pred_score_i columns (default -1 = not scored)
for i in range(1, 22):
    df[f"pred_score_{i}"] = -1  # -1 = not a relevant symptom / not scored

parse_errors = 0
for row_idx, raw in enumerate(raw_score_outputs):
    if isinstance(raw, str) and raw.startswith("[ERROR"):
        parse_errors += 1
        continue
    try:
        parsed = json.loads(raw)
        for item_str, score_val in parsed.get("scores", {}).items():
            item_num = int(item_str)
            score_val = int(score_val)
            if 1 <= item_num <= 21 and 0 <= score_val <= 3:
                df.at[row_idx, f"pred_score_{item_num}"] = score_val
    except (json.JSONDecodeError, ValueError, TypeError):
        parse_errors += 1

print(f"\nParse errors: {parse_errors} / {len(df)}")

df.to_csv("gpt_bdi_score_predictions.csv", index=False)
print("Saved → gpt_bdi_score_predictions.csv")

Loaded 2332 rows.


GPT score detection: 100%|██████████| 2332/2332 [20:08<00:00,  1.93it/s]


Parse errors: 0 / 2332
Saved → gpt_bdi_score_predictions.csv


In [5]:
from sklearn.metrics import mean_absolute_error, accuracy_score, cohen_kappa_score, mean_squared_error
from scipy.stats import spearmanr
import numpy as np

results = []
all_gt_parts, all_pred_parts = [], []

for i in range(1, 22):
    gt_col, pred_col = f"{i}.Score", f"pred_score_{i}"
    mask = (df[gt_col] >= 0) & (df[pred_col] >= 0)
    if mask.sum() == 0:
        continue
    gt   = df.loc[mask, gt_col].astype(int).values
    pred = df.loc[mask, pred_col].astype(int).values

    # Accumulate for overall metrics
    all_gt_parts.append(gt)
    all_pred_parts.append(pred)

    # rho = spearmanr(gt, pred).statistic if len(set(gt)) > 1 else float("nan")
    qwk = cohen_kappa_score(gt, pred, weights="quadratic") if len(set(gt)) > 1 else float("nan")

    results.append({
        "Item"      : i,
        "Symptom"   : SYMPTOMS[i],
        "N"         : int(mask.sum()),
        "Exact Acc" : round(accuracy_score(gt, pred), 3),
        "Within-1"  : round((np.abs(gt - pred) <= 1).mean(), 3),
        "MAE"       : round(mean_absolute_error(gt, pred), 3),
        "RMSE"      : round(np.sqrt(mean_squared_error(gt, pred)), 3),
        "QWK"       : round(qwk, 3),
    })

eval_df = pd.DataFrame(results)
print(eval_df.to_string(index=False))

# ── Overall ───────────────────────────────────────────────────────────────────
all_gt   = np.concatenate(all_gt_parts).astype(int)
all_pred = np.concatenate(all_pred_parts).astype(int)

rho_all = spearmanr(all_gt, all_pred).statistic
print(f"\nOverall Exact Acc : {accuracy_score(all_gt, all_pred):.4f}")
print(f"Overall Within-1  : {(np.abs(all_gt - all_pred) <= 1).mean():.4f}")
print(f"Overall MAE       : {mean_absolute_error(all_gt, all_pred):.4f}")
print(f"Overall RMSE      : {np.sqrt(mean_squared_error(all_gt, all_pred)):.4f}")
print(f"Overall QWK       : {cohen_kappa_score(all_gt, all_pred, weights='quadratic'):.4f}")

eval_df.to_csv("gpt_bdi_score_eval.csv", index=False)
print("\nSaved → gpt_bdi_score_eval.csv")

 Item                     Symptom   N  Exact Acc  Within-1   MAE  RMSE   QWK
    1                     Sadness 156      0.423     0.795 0.808 1.149 0.322
    2                   Pessimism 177      0.463     0.842 0.712 1.047 0.400
    3                Past Failure 148      0.378     0.818 0.824 1.127 0.353
    4            Loss of Pleasure 129      0.380     0.876 0.775 1.071 0.412
    5             Guilty Feelings 156      0.333     0.750 0.955 1.268 0.205
    6         Punishment Feelings 117      0.496     0.812 0.726 1.113 0.370
    7                Self-Dislike 128      0.445     0.914 0.656 0.944 0.528
    8           Self-Criticalness 193      0.363     0.762 0.891 1.196 0.225
    9 Suicidal Thoughts or Wishes 113      0.761     0.903 0.345 0.758 0.571
   10                      Crying 106      0.566     0.877 0.585 0.971 0.577
   11                   Agitation 132      0.447     0.856 0.697 0.992 0.422
   12            Loss of Interest 140      0.400     0.843 0.764 1.052 0.433

This paper reports lower: https://link.springer.com/chapter/10.1007/978-3-031-98420-*4_17*

# Train SBERT

In [ ]:
import sys
mods_to_remove = [k for k in sys.modules if "torchvision" in k]
for m in mods_to_remove:
    del sys.modules[m]
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False

print("Patch applied:", ds_config.TORCHVISION_AVAILABLE)  # should print False

!pip install -q transformers datasets scikit-learn coral-pytorch accelerate

import sys
for m in [k for k in sys.modules if "torchvision" in k]:
    del sys.modules[m]
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False

# Fix torchvision/datasets conflict before anything else
import sys, os

# Prevent the VideoReader import bug in torchvision
if "torchvision.io" in sys.modules:
    del sys.modules["torchvision.io"]

import warnings, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, mean_absolute_error
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from transformers import EarlyStoppingCallback


warnings.filterwarnings("ignore")

# CONFIG
MODE       = "ce"          # "ce" (cross-entropy) or "coral" (ordinal)
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
DATA_PATH  = "bdichat_turn_annotations.csv"
MAX_LEN    = 256
BATCH_SIZE = 16
GRAD_ACCUM = 2             # effective batch size = 32
EPOCHS     = 20
PATIENCE   = 4
LR         = 2e-5
SEED       = 42
OUTPUT_DIR = "./sbert_bdi_score_model"

SYMPTOMS = {
    1: "Sadness", 2: "Pessimism", 3: "Past Failure", 4: "Loss of Pleasure",
    5: "Guilty Feelings", 6: "Punishment Feelings", 7: "Self-Dislike",
    8: "Self-Criticalness", 9: "Suicidal Thoughts or Wishes", 10: "Crying",
    11: "Agitation", 12: "Loss of Interest", 13: "Indecisiveness",
    14: "Worthlessness", 15: "Loss of Energy", 16: "Changes in Sleeping Pattern",
    17: "Irritability", 18: "Changes in Appetite", 19: "Concentration Difficulty",
    20: "Tiredness or Fatigue", 21: "Loss of Interest in Sex",
}

# Items 16 & 18: raw 0–6 (0, 1a, 1b, 2a, 2b, 3a, 3b) → collapse to 0–3
AB_ITEMS = {16, 18}
AB_MAP   = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3, 6: 3}

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["PersonaReply"]).reset_index(drop=True)
df["Message"] = df["Message"].fillna("")

records = []
for _, row in df.iterrows():
    for i in range(1, 22):
        score_col = f"{i}.Score"
        if score_col not in df.columns:
            continue
        raw = row[score_col]
        if pd.isna(raw) or raw < 0:
            continue
        score = AB_MAP[int(raw)] if i in AB_ITEMS else int(raw)
        records.append({
            "text":  f"{SYMPTOMS[i]}: {row['Message']} [SEP] {row['PersonaReply']}",
            "label": score,
            "item":  i,
        })

print(f"Total examples: {len(records)}")
print(pd.Series([r['label'] for r in records]).value_counts().sort_index().to_string())

# TRAIN / VAL SPLIT
train_recs, val_recs = train_test_split(
    records, test_size=0.2, random_state=32,
    stratify=[r["label"] for r in records],
)
print(f"\nTrain: {len(train_recs)}  |  Val: {len(val_recs)}")

# TOKENISATION
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"], truncation=True,
        padding="max_length", max_length=MAX_LEN,
    )

train_ds = Dataset.from_list(train_recs).map(tokenize, batched=True)
val_ds   = Dataset.from_list(val_recs).map(tokenize, batched=True)

# use .with_format() to avoid torchvision VideoReader import
train_ds = train_ds.with_format("torch")
val_ds   = val_ds.with_format("torch")

# METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if MODE == "coral":
        preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).sum(dim=1).numpy().astype(int)
    else:
        preds = np.argmax(logits, axis=-1)
    labels = labels.astype(int)
    qwk = cohen_kappa_score(labels, preds, weights="quadratic") \
          if len(set(labels)) > 1 else float("nan")
    return {
        "qwk":       round(float(qwk), 4),
        "exact_acc": round(float(accuracy_score(labels, preds)), 4),
        "within1":   round(float((np.abs(preds - labels) <= 1).mean()), 4),
        "mae":       round(float(mean_absolute_error(labels, preds)), 4),
    }

# MODEL
total_steps   = (len(train_recs) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps  = int(0.1 * total_steps)

if MODE == "ce":
    print("\nMode: cross-entropy (4-class)")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

else:
    print("\nMode: ordinal CORAL")
    from coral_pytorch.losses import corn_loss
    from coral_pytorch.layers import CoralLayer

    class BDIScoreModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(MODEL_NAME)
            hidden = self.encoder.config.hidden_size
            self.coral = CoralLayer(hidden, num_classes=4)

        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            logits = self.coral(out.pooler_output)
            loss   = corn_loss(logits, labels.long(), num_classes=4) if labels is not None else None
            return {"loss": loss, "logits": logits}

    model = BDIScoreModel()

# TRAINING ARGS
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_steps=warmup_steps,             # ← replaces deprecated warmup_ratio
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=(MODE == "ce"),  # must be False for custom CORAL model
    seed=SEED,
    report_to="none",
)

# TRAIN
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("\nStarting training …\n")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel saved → {OUTPUT_DIR}/")
print(json.dumps(trainer.evaluate(), indent=2))

Patch applied: False
Total examples: 2845
0    616
1    868
2    792
3    569

Train: 2276  |  Val: 569


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (21 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved → ./sbert_bdi_score_model/


Training Loss,Validation Loss,Epoch,Qwk,Exact Acc,Within1,Mae
0.980222,1.098666,12,0.601200,0.543100,0.884000,0.595800


{
  "eval_loss": 1.0986663103103638,
  "eval_qwk": 0.6012,
  "eval_exact_acc": 0.5431,
  "eval_within1": 0.884,
  "eval_mae": 0.5958
}


In [7]:
from google.colab import drive
drive.mount('/content/drive')
import os
import shutil

source_dir = './sbert_bdi_score_model'
dest_dir = '/content/drive/MyDrive/sbert_bdi_score_model'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
    print(f"Contents of '{source_dir}' moved to '{dest_dir}'")
else:
    print(f"Source directory '{source_dir}' does not exist.")

Mounted at /content/drive
Contents of './sbert_bdi_score_model' moved to '/content/drive/MyDrive/sbert_bdi_score_model'


# Train MentalBERT

In [ ]:
import sys
mods_to_remove = [k for k in sys.modules if "torchvision" in k]
for m in mods_to_remove:
    del sys.modules[m]
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False

print("Patch applied:", ds_config.TORCHVISION_AVAILABLE)  # should print False

!pip install -q transformers datasets scikit-learn coral-pytorch accelerate

import sys
for m in [k for k in sys.modules if "torchvision" in k]:
    del sys.modules[m]
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False

# Fix torchvision/datasets conflict before anything else
import sys, os

# Prevent the VideoReader import bug in torchvision
if "torchvision.io" in sys.modules:
    del sys.modules["torchvision.io"]

import warnings, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, mean_absolute_error
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from transformers import EarlyStoppingCallback


warnings.filterwarnings("ignore")

# CONFIG
MODE       = "ce"          # "ce" (cross-entropy) or "coral" (ordinal)
MODEL_NAME = "mental/mental-bert-base-uncased"
DATA_PATH  = "bdichat_turn_annotations.csv"
MAX_LEN    = 256
BATCH_SIZE = 16
GRAD_ACCUM = 2             # effective batch size = 32
EPOCHS     = 20
PATIENCE   = 4
LR         = 2e-5
SEED       = 42
OUTPUT_DIR = "./mentalbert_bdi_score_model"

SYMPTOMS = {
    1: "Sadness", 2: "Pessimism", 3: "Past Failure", 4: "Loss of Pleasure",
    5: "Guilty Feelings", 6: "Punishment Feelings", 7: "Self-Dislike",
    8: "Self-Criticalness", 9: "Suicidal Thoughts or Wishes", 10: "Crying",
    11: "Agitation", 12: "Loss of Interest", 13: "Indecisiveness",
    14: "Worthlessness", 15: "Loss of Energy", 16: "Changes in Sleeping Pattern",
    17: "Irritability", 18: "Changes in Appetite", 19: "Concentration Difficulty",
    20: "Tiredness or Fatigue", 21: "Loss of Interest in Sex",
}

# Items 16 & 18: raw 0–6 (0, 1a, 1b, 2a, 2b, 3a, 3b) → collapse to 0–3
AB_ITEMS = {16, 18}
AB_MAP   = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3, 6: 3}

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["PersonaReply"]).reset_index(drop=True)
df["Message"] = df["Message"].fillna("")

records = []
for _, row in df.iterrows():
    for i in range(1, 22):
        score_col = f"{i}.Score"
        if score_col not in df.columns:
            continue
        raw = row[score_col]
        if pd.isna(raw) or raw < 0:
            continue
        score = AB_MAP[int(raw)] if i in AB_ITEMS else int(raw)
        records.append({
            "text":  f"{SYMPTOMS[i]}: {row['Message']} [SEP] {row['PersonaReply']}",
            "label": score,
            "item":  i,
        })

print(f"Total examples: {len(records)}")
print(pd.Series([r['label'] for r in records]).value_counts().sort_index().to_string())

# TRAIN / VAL SPLIT
train_recs, val_recs = train_test_split(
    records, test_size=0.2, random_state=32,
    stratify=[r["label"] for r in records],
)
print(f"\nTrain: {len(train_recs)}  |  Val: {len(val_recs)}")

# TOKENISATION
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"], truncation=True,
        padding="max_length", max_length=MAX_LEN,
    )

train_ds = Dataset.from_list(train_recs).map(tokenize, batched=True)
val_ds   = Dataset.from_list(val_recs).map(tokenize, batched=True)

# use .with_format() to avoid torchvision VideoReader import
train_ds = train_ds.with_format("torch")
val_ds   = val_ds.with_format("torch")

# METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if MODE == "coral":
        preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).sum(dim=1).numpy().astype(int)
    else:
        preds = np.argmax(logits, axis=-1)
    labels = labels.astype(int)
    qwk = cohen_kappa_score(labels, preds, weights="quadratic") \
          if len(set(labels)) > 1 else float("nan")
    return {
        "qwk":       round(float(qwk), 4),
        "exact_acc": round(float(accuracy_score(labels, preds)), 4),
        "within1":   round(float((np.abs(preds - labels) <= 1).mean()), 4),
        "mae":       round(float(mean_absolute_error(labels, preds)), 4),
    }

# MODEL
total_steps   = (len(train_recs) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps  = int(0.1 * total_steps)

if MODE == "ce":
    print("\nMode: cross-entropy (4-class)")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

else:
    print("\nMode: ordinal CORAL")
    from coral_pytorch.losses import corn_loss
    from coral_pytorch.layers import CoralLayer

    class BDIScoreModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(MODEL_NAME)
            hidden = self.encoder.config.hidden_size
            self.coral = CoralLayer(hidden, num_classes=4)

        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            logits = self.coral(out.pooler_output)
            loss   = corn_loss(logits, labels.long(), num_classes=4) if labels is not None else None
            return {"loss": loss, "logits": logits}

    model = BDIScoreModel()

# TRAINING ARGS
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_steps=warmup_steps,             # ← replaces deprecated warmup_ratio
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=(MODE == "ce"),  # must be False for custom CORAL model
    seed=SEED,
    report_to="none",
)

# TRAIN
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("\nStarting training …\n")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel saved → {OUTPUT_DIR}/")
print(json.dumps(trainer.evaluate(), indent=2))

Patch applied: False
Total examples: 2845
0    616
1    868
2    792
3    569

Train: 2276  |  Val: 569


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (21 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved → ./mentalbert_bdi_score_model/


Training Loss,Validation Loss,Epoch,Qwk,Exact Acc,Within1,Mae
0.561874,1.174080,10,0.638000,0.574700,0.926200,0.509700


{
  "eval_loss": 1.174080491065979,
  "eval_qwk": 0.638,
  "eval_exact_acc": 0.5747,
  "eval_within1": 0.9262,
  "eval_mae": 0.5097
}


In [9]:
from google.colab import drive
drive.mount('/content/drive')
import os
import shutil

source_dir = './mentalbert_bdi_score_model'
dest_dir = '/content/drive/MyDrive/mentalbert_bdi_score_model'

def ignore_checkpoints(dir_path, contents):
    return [c for c in contents if c.startswith('checkpoint-')]

if os.path.exists(source_dir):
    shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True, ignore=ignore_checkpoints)
    print(f"Contents of '{source_dir}' (excluding checkpoints) copied to '{dest_dir}'")
else:
    print(f"Source directory '{source_dir}' does not exist.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Contents of './mentalbert_bdi_score_model' (excluding checkpoints) copied to '/content/drive/MyDrive/mentalbert_bdi_score_model'


# Eval

In [10]:
# Evaluate fine-tuned model + GPT on the TEST SET

import sys, warnings, json
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score,
    mean_absolute_error, mean_squared_error,
    f1_score,
)
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# CONFIG
# Map each fine-tuned model's display name -> its saved checkpoint directory.
MODEL_DIRS = {
    "SBERT":      "./sbert_bdi_score_model",
    "MentalBERT": "./mentalbert_bdi_score_model",
}
DATA_PATH  = "bdichat_turn_annotations.csv"
GPT_PREDS  = "gpt_bdi_score_predictions.csv"
MAX_LEN    = 256
BATCH_SIZE = 32
SEED       = 42

SYMPTOMS = {
    1: "Sadness", 2: "Pessimism", 3: "Past Failure", 4: "Loss of Pleasure",
    5: "Guilty Feelings", 6: "Punishment Feelings", 7: "Self-Dislike",
    8: "Self-Criticalness", 9: "Suicidal Thoughts or Wishes", 10: "Crying",
    11: "Agitation", 12: "Loss of Interest", 13: "Indecisiveness",
    14: "Worthlessness", 15: "Loss of Energy", 16: "Changes in Sleeping Pattern",
    17: "Irritability", 18: "Changes in Appetite", 19: "Concentration Difficulty",
    20: "Tiredness or Fatigue", 21: "Loss of Interest in Sex",
}
AB_ITEMS = {16, 18}
AB_MAP   = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 3, 6: 3}

# REBUILD RECORDS
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["PersonaReply"]).reset_index(drop=True)
df["Message"] = df["Message"].fillna("")

records = []
for row_idx, row in df.iterrows():
    for i in range(1, 22):
        score_col = f"{i}.Score"
        if score_col not in df.columns:
            continue
        raw = row[score_col]
        if pd.isna(raw) or raw < 0:
            continue
        score = AB_MAP[int(raw)] if i in AB_ITEMS else int(raw)
        records.append({
            "text":    f"{SYMPTOMS[i]}: {row['Message']} [SEP] {row['PersonaReply']}",
            "label":   score,
            "item":    i,
            "row_idx": row_idx,
        })

# SPLIT
train_recs, test_recs = train_test_split(
    records, test_size=0.2, random_state=SEED,
    stratify=[r["label"] for r in records],
)
print(f"Test set: {len(test_recs)} examples")

test_row_idxs = [r["row_idx"] for r in test_recs]
test_items    = np.array([r["item"]  for r in test_recs])
test_labels   = np.array([r["label"] for r in test_recs])

# FINE-TUNED MODEL INFERENCE (SBERT + MentalBERT)
# Each model has its own tokenizer/vocab, so the test set is re-tokenized per model.
model_preds = {}  # display name -> np.array of predicted class indices

for model_name, model_dir in MODEL_DIRS.items():
    print(f"\nLoading {model_name} from {model_dir} …")
    tok = AutoTokenizer.from_pretrained(model_dir)

    def tokenize(batch, tok=tok):
        return tok(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

    ds = Dataset.from_list(test_recs).map(tokenize, batched=True).with_format("torch")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir, num_labels=4)
    model.eval()

    eval_args = TrainingArguments(
        output_dir=f"./tmp_eval_{model_name.lower()}",
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    trainer = Trainer(model=model, args=eval_args)
    model_preds[model_name] = np.argmax(trainer.predict(ds).predictions, axis=-1)
    print(f"{model_name} loaded and predicted on {len(test_recs)} test examples")

# GPT PREDICTIONS filter to test set
gpt_df = pd.read_csv(GPT_PREDS)

gpt_preds = []
for r in test_recs:
    col = f"pred_score_{r['item']}"
    val = gpt_df.at[r["row_idx"], col] if col in gpt_df.columns else -1
    gpt_preds.append(int(val))
gpt_preds = np.array(gpt_preds)

gpt_mask = gpt_preds >= 0
print(f"GPT scored {gpt_mask.sum()} / {len(test_recs)} test examples")

# EVALUATION HELPER
def eval_metrics(gt, pred):
    qwk = cohen_kappa_score(gt, pred, weights="quadratic") if len(set(gt)) > 1 else float("nan")
    # F1: use weighted to account for class imbalance; labels=all 4 classes
    all_labels = [0, 1, 2, 3]
    f1_macro    = f1_score(gt, pred, average="macro",    labels=all_labels, zero_division=0)
    f1_weighted = f1_score(gt, pred, average="weighted", labels=all_labels, zero_division=0)
    mse         = mean_squared_error(gt, pred)            # MSE (not rooted)
    return {
        "N":           len(gt),
        "Exact Acc":   round(accuracy_score(gt, pred), 3),
        "Within-1":    round((np.abs(gt - pred) <= 1).mean(), 3),
        "MAE":         round(mean_absolute_error(gt, pred), 3),
        "MSE":         round(mse, 3),
        "RMSE":        round(np.sqrt(mse), 3),
        "F1 (macro)":  round(f1_macro, 3),
        "F1 (wtd)":    round(f1_weighted, 3),
        "QWK":         round(qwk, 3),
    }

# PER-SYMPTOM COMPARISON
rows = []
for i in range(1, 22):
    mask = test_items == i

    for model_name, preds in model_preds.items():
        m = eval_metrics(test_labels[mask], preds[mask])
        rows.append({"Source": model_name, "Item": i, "Symptom": SYMPTOMS[i], **m})

    g_mask = mask & gpt_mask
    if g_mask.sum() > 0:
        g = eval_metrics(test_labels[g_mask], gpt_preds[g_mask])
        rows.append({"Source": "GPT", "Item": i, "Symptom": SYMPTOMS[i], **g})

cmp_df = pd.DataFrame(rows)
print("\n── Per-symptom results ──")
print(cmp_df.to_string(index=False))

# OVERALL
print("\n── Overall ──")
overall_sources = [
    (model_name, test_labels, preds, np.ones(len(test_labels), dtype=bool))
    for model_name, preds in model_preds.items()
] + [
    ("GPT (scored only)", test_labels[gpt_mask], gpt_preds[gpt_mask], gpt_mask),
]

for label, gt, pred, mask in overall_sources:
    m = eval_metrics(gt[mask[:len(gt)]], pred[mask[:len(pred)]])
    print(f"\n{label}  (N={m['N']})")
    print(f"  Exact Acc  : {m['Exact Acc']}")
    print(f"  Within-1   : {m['Within-1']}")
    print(f"  MAE        : {m['MAE']}")
    print(f"  MSE        : {m['MSE']}")
    print(f"  RMSE       : {m['RMSE']}")
    print(f"  F1 (macro) : {m['F1 (macro)']}")
    print(f"  F1 (wtd)   : {m['F1 (wtd)']}")
    print(f"  QWK        : {m['QWK']}")

cmp_df.to_csv("bdi_score_eval_comparison.csv", index=False)
print("\nSaved → bdi_score_eval_comparison.csv")

Test set: 569 examples

Loading SBERT from ./sbert_bdi_score_model …


Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

SBERT loaded and predicted on 569 test examples

Loading MentalBERT from ./mentalbert_bdi_score_model …


Map:   0%|          | 0/569 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MentalBERT loaded and predicted on 569 test examples
GPT scored 568 / 569 test examples

── Per-symptom results ──
    Source  Item                     Symptom  N  Exact Acc  Within-1   MAE   MSE  RMSE  F1 (macro)  F1 (wtd)   QWK
     SBERT     1                     Sadness 35      0.600     0.943 0.457 0.571 0.756       0.581     0.540 0.763
MentalBERT     1                     Sadness 35      0.600     0.914 0.514 0.800 0.894       0.608     0.591 0.605
       GPT     1                     Sadness 35      0.371     0.800 0.857 1.371 1.171       0.347     0.312 0.356
     SBERT     2                   Pessimism 38      0.474     0.895 0.658 0.974 0.987       0.443     0.469 0.632
MentalBERT     2                   Pessimism 38      0.500     0.895 0.605 0.816 0.903       0.516     0.527 0.609
       GPT     2                   Pessimism 38      0.500     0.895 0.605 0.816 0.903       0.408     0.453 0.556
     SBERT     3                Past Failure 26      0.500     0.692 0.808 1.423